# Единая методика финансового эффекта

Генератор артефактов в `data/`:

1. `fin_effect_plan.html` — план и бизнес-схема;
2. `fin_effect_report.html` — расчёт, доли путей, формулы;
3. `fin_effect_conclusion.html` — заключение по расчёту;
4. `fin_effect_audit.xlsx` — Excel-аудит (вводные + формулы; bootstrap — значения из Python).

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import pandas as pd

_here = Path.cwd().resolve()
PROJECT_ROOT = next(
    path for path in (_here, *_here.parents) if (path / "pyproject.toml").exists()
)
for path in (PROJECT_ROOT / "src", PROJECT_ROOT):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from querulus.fin_effect.excel_monitoring import (
    RETRO_AS_OF_DEFAULT,
    VITRINA_TABLE_DEFAULT,
    estimate_monitoring_effect,
    load_monitoring_frame,
)
from querulus.fin_effect.monitoring_excel_audit import export_monitoring_audit_xlsx
from querulus.fin_effect.monitoring_report import (
    REPORT_FILENAME,
    write_all_monitoring_htmls,
    write_error_html,
    write_plan_html,
)
from querulus.fin_effect.psr_development import compute_psr_development_lags

DATA_DIR = PROJECT_ROOT / "monitoring" / "fin_effects" / "data"
REPORT_PATH = DATA_DIR / REPORT_FILENAME
AUDIT_XLSX = DATA_DIR / "fin_effect_audit.xlsx"
RETRO_PARQUET = Path(
    "/home/jovyan/old_home/querulus/data/processed/querulus_train_dataset.parquet"
)
PRETENSIONS_PARQUET = Path(
    "/home/jovyan/old_home/querulus/data/raw/df_pretensions.parquet"
)
CLAIMS_PARQUET = Path(
    "/home/jovyan/old_home/querulus/data/raw/df_claims_incoming.parquet"
)
LOOKBACK_YEARS = 2.0
RETRO_AS_OF = RETRO_AS_OF_DEFAULT
T_CALC = None
DISCOUNT_RATE = 0.12
RESIDUAL_SHARE = 0.07
BOOTSTRAP_ITERATIONS = 1000
BOOTSTRAP_COMPLIANCE_ITERATIONS = 200
# -1 = все ядра кроме одного; 1 = без параллели
BOOTSTRAP_N_JOBS = -1
WRITE_AUDIT_XLSX = True

In [ ]:
plan_path = write_plan_html(
    DATA_DIR / "fin_effect_plan.html",
    source_label=VITRINA_TABLE_DEFAULT,
)
report_path = REPORT_PATH
conclusion_path = DATA_DIR / "fin_effect_conclusion.html"
audit_path = AUDIT_XLSX

try:
    monitoring_df = load_monitoring_frame(
        source="mssql",
        table=VITRINA_TABLE_DEFAULT,
    )
    if not RETRO_PARQUET.exists():
        raise FileNotFoundError(
            f"Не найден финальный ретро-датасет: {RETRO_PARQUET}"
        )
    retro_df = pd.read_parquet(RETRO_PARQUET)
    result = estimate_monitoring_effect(
        monitoring_df,
        retro_df,
        t_calc=T_CALC,
        residual_share=RESIDUAL_SHARE,
        discount_rate=DISCOUNT_RATE,
        lookback_years=LOOKBACK_YEARS,
        retro_as_of=RETRO_AS_OF,
        bootstrap_iterations=BOOTSTRAP_ITERATIONS,
        bootstrap_compliance_iterations=BOOTSTRAP_COMPLIANCE_ITERATIONS,
        bootstrap_n_jobs=BOOTSTRAP_N_JOBS,
    )
    development_lags = None
    pret = (
        pd.read_parquet(PRETENSIONS_PARQUET)
        if PRETENSIONS_PARQUET.exists()
        else None
    )
    claims = pd.read_parquet(CLAIMS_PARQUET) if CLAIMS_PARQUET.exists() else None
    if pret is not None or claims is not None:
        development_lags = compute_psr_development_lags(
            pretensions=pret,
            claims=claims,
            incidents=retro_df,
        )
    plan_path, report_path, conclusion_path = write_all_monitoring_htmls(
        result,
        DATA_DIR,
        source_label=VITRINA_TABLE_DEFAULT,
        development_lags=development_lags,
    )
    if WRITE_AUDIT_XLSX:
        audit_path = export_monitoring_audit_xlsx(result, AUDIT_XLSX)
except Exception as exc:
    report_path = write_error_html(
        exc,
        REPORT_PATH,
        source_label=VITRINA_TABLE_DEFAULT,
    )
    raise
finally:
    print(f"План → {plan_path}")
    print(f"Расчёт → {report_path}")
    print(f"Заключение → {conclusion_path}")
    print(f"Excel-аудит → {audit_path}")
